In [102]:
import pandas as pd
import numpy as np

In [103]:
df=pd.read_csv('queries_train.csv')

In [104]:
df.head()

,query_id,query,relevant_article_id,category,title,body
0,Q1001,why set trial lentgh to 14 days asap,A121,Subscriptions,Free trials,Subscriptions can start with a free trial of a...
1,Q1002,how do i fight a chargeback?,A105,Disputes,Responding to a dispute,When a customer disputes a charge with their b...
2,Q1003,how do i upload id to keep my account active?,A116,Account,Verifying your business identity,"To keep payouts enabled, we need details such ..."
3,Q1004,"hi, dunning settings for subscriptions",A122,Subscriptions,Retrying failed subscription payments,"If a renewal payment fails, it is retried auto..."
4,Q1005,need help: upgrade a subscriber mid month on m...,A120,Subscriptions,Changing a customer's subscription plan,"When a customer upgrades or downgrades, the pr..."


In [105]:
df.head()

,query_id,query,relevant_article_id,category,title,body
0,Q1001,why set trial lentgh to 14 days asap,A121,Subscriptions,Free trials,Subscriptions can start with a free trial of a...
1,Q1002,how do i fight a chargeback?,A105,Disputes,Responding to a dispute,When a customer disputes a charge with their b...
2,Q1003,how do i upload id to keep my account active?,A116,Account,Verifying your business identity,"To keep payouts enabled, we need details such ..."
3,Q1004,"hi, dunning settings for subscriptions",A122,Subscriptions,Retrying failed subscription payments,"If a renewal payment fails, it is retried auto..."
4,Q1005,need help: upgrade a subscriber mid month on m...,A120,Subscriptions,Changing a customer's subscription plan,"When a customer upgrades or downgrades, the pr..."


In [106]:
# 1. Drop duplicates to get ONLY the 24 unique Stripe help articles
articles_df = (
    df[["relevant_article_id", "category", "title", "body"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 2. Combine ONLY article fields (NO customer queries here!)
articles_df["combined_text"] = (
    articles_df["relevant_article_id"]
    + " "
    + articles_df["category"]
    + " "
    + articles_df["title"]
    + " "
    + articles_df["body"]
)

# 3. Build your tokenized corpus and models from these 24 articles
tokenized_corpus = [doc.lower().split() for doc in articles_df["combined_text"]]

In [107]:
import sys
if 'gensim' not in sys.modules:
  !pip install gensim
if 'rank_bm25' not in sys.modules:
  !pip install rank_bm25

from gensim.models import Word2Vec
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [108]:
bm25 = BM25Okapi(tokenized_corpus)

In [109]:
w2vec=Word2Vec(tokenized_corpus,vector_size=100,window=5,min_count=1,workers=5)

In [110]:
def get_document_vector(tokenized_text,model):
  vectors=[

           model.wv[word] for word in tokenized_text if word in model.wv
  ]

  if not vectors:
    return np.zeros(model.vector_size)
  return np.mean(vectors,axis=0)

In [111]:
article_vectors=np.array([get_document_vector(doc,w2vec) for doc in tokenized_corpus])

In [112]:
def hybrid_search(query_text,top_k=3,alpha=0.5):
  query_token=query_text.lower().split()

  bm25_scores=bm25.get_scores(query_token)

  query_vector=get_document_vector(query_token,w2vec)

  w2v_scores=cosine_similarity([query_vector],article_vectors).flatten()

  scaler=MinMaxScaler()
  if bm25_scores.max()>bm25_scores.min():
    bm25_norm=scaler.fit_transform(bm25_scores.reshape(-1,1)).flatten()
  else:
    bm25_norm=bm25_scores

  # Move w2v_norm calculation out of the bm25_scores else block
  if w2v_scores.max()>w2v_scores.min():
    w2v_norm=scaler.fit_transform(w2v_scores.reshape(-1,1)).flatten()
  else:
    w2v_norm=w2v_scores

  hybrid_scores=(alpha*bm25_norm)+((1-alpha)*w2v_norm)

  top_indices=hybrid_scores.argsort()[::-1][:top_k]

  results=[]

  for idx in top_indices:
    results.append({
        "article_id": row["relevant_article_id"],
        "category": row["category"],
        "title": row["title"],
        "body": row["body"],
    })
  return results

In [113]:
def evaluate_search(test_dataset):
  recall_hits = 0
  mrr_sum = 0.0
  total_queries = len(test_dataset)

  for item in test_dataset:
    query = item["customer_query"]
    true_code = item["true_article_code"]

    top_results = hybrid_search(query, top_k=3)
    predicted_codes = [res["article_code"] for res in top_results]

    # Recall@3 Check
    if true_code in predicted_codes:
      recall_hits += 1
      # MRR Check
      rank = predicted_codes.index(true_code) + 1
      mrr_sum += 1.0 / rank

  return {
      "Recall@3": recall_hits / total_queries,
      "MRR": mrr_sum / total_queries,
  }

In [114]:
test_df = pd.read_csv("queries_test.csv")
predictions = []

for _, row in test_df.iterrows():
  q_id = row["query_id"]
  q_text = row["query"]

  top_3 = hybrid_search(q_text, top_k=3)

  predictions.append({
      "query_id": q_id,
      "query": q_text,
      "rank_1_article": top_3[0]["article_code"],
      "rank_1_title": top_3[0]["title"],
      "rank_2_article": top_3[1]["article_code"],
      "rank_2_title": top_3[1]["title"],
      "rank_3_article": top_3[2]["article_code"],
      "rank_3_title": top_3[2]["title"],
  })

output_df = pd.DataFrame(predictions)
output_df.to_csv("test_predictions.csv", index=False)
print("Saved top 3 predictions for all test queries!")

Saved top 3 predictions for all test queries!


In [115]:
test_check_df=pd.read_csv('test_predictions.csv')
test_check_df.head()

,query_id,query,rank_1_article,rank_1_title,rank_2_article,rank_2_title,rank_3_article,rank_3_title
0,Q5001,help extra security code at sign in,A117,Adding team members and roles,A112,Refunding a customer,A105,Responding to a dispute
1,Q5002,fee for instant transfer,A117,Adding team members and roles,A104,Instant payouts to a debit card,A108,Why was a payment declined?
2,Q5003,why take payment after shipping,A114,Understanding processing fees,A112,Refunding a customer,A106,Dispute fees and when they are refunded
3,Q5004,refund bounced because card was cancelled,A111,Authorize now and capture later,A101,When will I receive my payout?,A119,Closing your account
4,Q5005,why switch pricing tier for a user on my account,A117,Adding team members and roles,A114,Understanding processing fees,A101,When will I receive my payout?
